# Handling Skewed Data

Skewed data happens when certain keys or groups are disproportionately represented in the dataset, leading to unbalanced workloads.
Dealing with skewed data in spark is essential for ensuring that tasks are evenly distributed across partitions, thus avoiding performance bottlenecks.

In [0]:
import time

## Salting

### Example 1:

#### Create Skewed Dataset

In [0]:
from pyspark.sql.functions import lit

customers = (
    spark.range(3000)
         .withColumn(
             "CustomerID",
             expr("""
                 CASE
                     WHEN id < 1000 THEN 'C1'
                     WHEN id < 2000 THEN 'C2'
                     ELSE 'C3'
                 END
             """)
         )
         .withColumn("CustomerName", lit("Customer"))
)

customers.count()

In [0]:
from pyspark.sql.functions import expr

orders = (
    spark.range(10000000)
         .withColumn(
             "CustomerID",
             expr("""
                 CASE
                     WHEN id < 9000000 THEN 'C1'
                     WHEN id < 9500000 THEN 'C2'
                     ELSE 'C3'
                 END
             """)
         )
)

orders.count()

In [0]:
spark.conf.get("spark.sql.shuffle.partitions")

In [0]:
orders = orders.repartition(10)
customers = customers.repartition(10)

In [0]:
from pyspark.sql.functions import spark_partition_id
orders.withColumn("partition_id", spark_partition_id()).groupBy("partition_Id").count().show(10)

In [0]:
joined_df = (
    orders.hint("MERGE")
          .join(
              customers.hint("MERGE"),
              "CustomerID"
          )
)

In [0]:
joined_df.explain("formatted")

In [0]:
from pyspark.sql.functions import floor, rand

orders_salted = orders.withColumn(
    "salt",
    floor(rand() * 10)
)

In [0]:
orders_salted.show(5)

In [0]:
from pyspark.sql.functions import explode
from pyspark.sql.functions import array
from pyspark.sql.functions import lit

customers_salted = customers.withColumn(
    "salt",
    explode(
        array(*[lit(i) for i in range(10)])
    )
)

In [0]:
customers_salted.show(5)

In [0]:
salted_join = (
    orders_salted.join(
        customers_salted,
        ["CustomerID", "salt"]
    )
)

In [0]:
salted_join.repartition(
    10,
    "CustomerID",
    "salt"
).withColumn(
    "partition_id",
    spark_partition_id()
).groupBy(
    "partition_id"
).count().orderBy(
    "partition_id"
).show(20, False)

#### Perform Join Without Salting

In [0]:
start = time.time()
joined_df.count()
print(time.time()-start)

#### Perform Join With Salting

In [0]:
start = time.time()
salted_join.count()
print(time.time()-start)

### Example 2:

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat, lit, rand

#### Sample Skewed DataFrame

In [0]:
sales_data = [
    ("India", 100),
    ("India", 200),
    ("India", 150),
    ("USA", 50),
    ("UK", 30)
]

sales_df = spark.createDataFrame(sales_data, ["country", "amount"])

country_data = [
    ("India", "Asia"),
    ("USA", "North America"),
    ("UK", "Europe")
]

country_df = spark.createDataFrame(country_data, ["country", "continent"])

#### Salting

SALTING: Add a random salt (0 to 2) to the sales_df

In [0]:
sales_salted = sales_df.withColumn("salt", (rand() * 3).cast("int")) \
    .withColumn("salted_key", concat(col("country"), lit("_"), col("salt")))

In [0]:
sales_salted.show()

Duplicate country_df for each salt value

In [0]:
salt_values = spark.createDataFrame([(0,), (1,), (2,)], ["salt"])
country_salted = country_df.crossJoin(salt_values) \
    .withColumn("salted_key", concat(col("country"), lit("_"), col("salt")))

In [0]:
country_salted.show()

#### Perform the salted join

In [0]:
sales_alias = sales_salted.alias("sales")
country_alias = country_salted.alias("country")

In [0]:
joined_df = sales_alias.join(country_alias, on="salted_key", how="inner")
print("🔹 Joined DataFrame:")
joined_df.show()

#### Select final columns clearly

In [0]:
final_result = joined_df.select(
    col("sales.country").alias("country"),
    col("sales.amount"),
    col("country.continent")
)
print("🔹 Final Result After Removing Salt:")
final_result.show()

### Example 3 (GroupBy/Aggregations)

## Broadcast Join

## Repartitioning

## Skew Join Optimization

## Custom Partitioning

## Avoid Shuffles

## Data Sampling